In [24]:
from __future__ import annotations

import argparse 
import json
import gzip
import re
import zipfile
from pathlib import Path
import typing

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from IPython.display import display

### Files' paths

In [25]:
ROOT_PATH = Path.cwd().parents[0]

RAW_DIR = ROOT_PATH / "data" / "raw"
SUPP_DIR = RAW_DIR / "lake_2024_supplement_data"

OUT_DIR = ROOT_PATH / "data" / "interim"
SUPP_OUT_DIR = OUT_DIR / "lake_2024_supplement_data"

META_DIR = ROOT_PATH / "data" / "metadata"

### Helpers

In [26]:
# Clean colnames
def clean_colname(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^0-9a-zA-Z_]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")

# Read tsv files
def read_table_tsv(path):
    with open(path, "r", errors="replace") as f:
        first_line = f.readline()
    
    sep = "\t" if first_line.count("\t") >= first_line.count(",") else ","
    df = pd.read_csv(path, sep=sep, low_memory=False)
    df.columns = [clean_colname(c) for c in df.columns]
    
    suffix = path.suffix.lower()
    
    df.columns = [clean_colname(c) for c in df.columns]
    
    return df


# Read vcf from bgz archive
def open_vcf(path):
    
    path = Path(path)
    suffixes = "".join(path.suffixes).lower()
    
    if suffixes.endswith(".gz") or suffixes.endswith(".bgz"):
        return gzip.open(path, "rt", errors="replace")
    
    return open(path, "r", errors="replace")

# INFO from vcf to dic
def parse_info(info):
    out = {}
    
    for item in info.split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            out[key] = value
        else:
            out[item] = True
    
    return out


### Define all files from Lake et al files

In [27]:
lake_files = []

lake_files.extend(sorted(SUPP_DIR.rglob("*.tsv")))
lake_files[0]

PosixPath('/Users/dmitriiiliushchenko/PhD/Projects/mtdna_constraint/data/raw/lake_2024_supplement_data/supplementary_dataset_1.tsv')

### Prepare Lake manifest and read all datasets to list

In [28]:
datasets = [None] * 10 # define list with 10 suupl data tables
manifest = [] # manifest file initiatiation

for path in lake_files:
    text = path.stem.lower()
    
    m = re.search(r"supplementary[_\s-]*dataset[_\s-]*(\d+)", text)
    
    if m is None:
        continue
    
    dataset_number = int(m.group(1))
    
    if dataset_number < 1 or dataset_number > 9:
        continue
    
    df = read_table_tsv(path)
    datasets[dataset_number] = df
    
    manifest.append({
        "dataset_number": dataset_number,
        "object_name": f"datasets[{dataset_number}]",
        "file": str(path),
        "n_rows": df.shape[0],
        "n_cols": df.shape[1],
        "columns": df.columns.tolist(),
    })

lake_manifest = pd.DataFrame(manifest).sort_values("dataset_number")

display(lake_manifest)

,dataset_number,object_name,file,n_rows,n_cols,columns
0,1,datasets[1],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,63,9,"[symbol, start_position, end_position, consequ..."
1,2,datasets[2],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,41,10,"[symbol, start_position, end_position, protein..."
2,3,datasets[3],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,171,4,"[position, reference, alternate, classificatio..."
3,4,datasets[4],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,75,6,"[trna_position, observed, expected, obs_exp, l..."
4,5,datasets[5],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,25,9,"[locus, description, start_position, end_posit..."
5,6,datasets[6],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,16568,2,"[position, mlc_pos_score]"
6,7,datasets[7],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,49704,5,"[position, reference, alternate, consequence, ..."
7,8,datasets[8],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,8284,2,"[variant, criteria]"
8,9,datasets[9],/Users/dmitriiiliushchenko/PhD/Projects/mtdna_...,88,5,"[variant, symbol, mitomap_plasmy, curated_stat..."


In [29]:
# Make shorter names
ds1, ds2, ds3, ds4, ds5, ds6, ds7, ds8, ds9 = datasets[1:10]

### Make variantID as in gnomad for suppl data

In [30]:
ds7.head()

,position,reference,alternate,consequence,mlc_score
0,1,G,T,intergenic_variant,0.65755
1,1,G,A,intergenic_variant,0.65755
2,1,G,C,intergenic_variant,0.65755
3,2,A,T,intergenic_variant,0.64832
4,2,A,C,intergenic_variant,0.64832


In [31]:
master = ds7.copy()

master["position"] = pd.to_numeric(master["position"], errors="coerce").astype("Int64")
master["reference"] = master["reference"].astype(str).str.upper()
master["alternate"] = master["alternate"].astype(str).str.upper()

master["variant_id"] = (
    "m."
    + master["position"].astype(str)
    + master["reference"]
    + ">"
    + master["alternate"]
)

display(master.head())
print(master.shape)
print(master["variant_id"].nunique())

,position,reference,alternate,consequence,mlc_score,variant_id
0,1,G,T,intergenic_variant,0.65755,m.1G>T
1,1,G,A,intergenic_variant,0.65755,m.1G>A
2,1,G,C,intergenic_variant,0.65755,m.1G>C
3,2,A,T,intergenic_variant,0.64832,m.2A>T
4,2,A,C,intergenic_variant,0.64832,m.2A>C


(49704, 6)
49704


In [32]:
ds6.head()

,position,mlc_pos_score
0,1,0.65755
1,2,0.64832
2,3,0.63872
3,4,0.63214
4,5,0.62584


In [33]:
ds6_clean = ds6.copy()

ds6_clean["position"] = pd.to_numeric(ds6_clean["position"], errors="coerce").astype("Int64")

ds6_clean = ds6_clean.rename(columns={
    "mlc_pos_score": "mlc_position_score"
})

master = master.merge(
    ds6_clean[["position", "mlc_position_score"]],
    on="position",
    how="left"
)

print(master.shape)
display(master.head())

(49704, 7)


,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832


Check if there is difference between mlc_score and position score

In [34]:
master[master.mlc_score != master.mlc_position_score]

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score
9915,3307,A,T,start_lost,0.7,m.3307A>T,0.33460
9916,3307,A,C,start_lost,0.7,m.3307A>C,0.33460
9917,3307,A,G,start_lost,0.7,m.3307A>G,0.33460
9918,3308,T,C,start_lost,0.7,m.3308T>C,0.29957
9919,3308,T,A,start_lost,0.7,m.3308T>A,0.29957
...,...,...,...,...,...,...,...
47653,15886,C,A,synonymous_variant,0.0,m.15886C>A,0.07547
47654,15886,C,G,synonymous_variant,0.0,m.15886C>G,0.07547
47655,15887,T,C,incomplete_terminal_codon_variant&coding_seque...,0.7,m.15887T>C,0.07526
47656,15887,T,A,incomplete_terminal_codon_variant&coding_seque...,0.7,m.15887T>A,0.07526


In [35]:
def parse_variant_string(x):
    
    x = str(x)
    
    # Get subs with position
    m = re.search(r"m\.(\d+)([ACGT])>([ACGT])", x, flags=re.IGNORECASE)
    
    if m is None:
        return pd.Series({
            "position": pd.NA,
            "reference": pd.NA,
            "alternate": pd.NA,
            "variant_id": pd.NA,
        })
    
    # Define
    position = int(m.group(1))
    reference = m.group(2).upper()
    alternate = m.group(3).upper()
    
    return pd.Series({
        "position": position,
        "reference": reference,
        "alternate": alternate,
        "variant_id": f"m.{position}{reference}>{alternate}",
    })

In [36]:
ds8.head()

,variant,criteria
0,m.10T>C,haplogroup_variant
1,m.16A>T,haplogroup_variant
2,m.26C>T,haplogroup_variant
3,m.41C>T,haplogroup_variant
4,m.47G>A,haplogroup_variant


In [37]:
ds8 = ds8.copy()

ds8[["position", "reference", "alternate", "variant_id"]] = (
    ds8["variant"].apply(parse_variant_string)
)

display(ds8.head())
print("shape:", ds8.shape)
print("unique variant_id:", ds8["variant_id"].nunique())

,variant,criteria,position,reference,alternate,variant_id
0,m.10T>C,haplogroup_variant,10,T,C,m.10T>C
1,m.16A>T,haplogroup_variant,16,A,T,m.16A>T
2,m.26C>T,haplogroup_variant,26,C,T,m.26C>T
3,m.41C>T,haplogroup_variant,41,C,T,m.41C>T
4,m.47G>A,haplogroup_variant,47,G,A,m.47G>A


shape: (8284, 6)
unique variant_id: 8284


In [38]:
ds9 = ds9.copy()

ds9[["position", "reference", "alternate", "variant_id"]] = (
    ds9["variant"].apply(parse_variant_string)
)

display(ds9.head())
print("shape:", ds9.shape)
print("missing variant_id:", ds9["variant_id"].isna().sum())
print("unique variant_id:", ds9["variant_id"].nunique())

,variant,symbol,mitomap_plasmy,curated_status,curated_homoplasmy_report,position,reference,alternate,variant_id
0,m.8969G>A,MT-ATP6,-/+,+/+,PMID:32042921,8969,G,A,m.8969G>A
1,m.8851T>C,MT-ATP6,+/+,+/+,NaN,8851,T,C,m.8851T>C
2,m.9185T>C,MT-ATP6,+/+,+/+,NaN,9185,T,C,m.9185T>C
3,m.9176T>C,MT-ATP6,+/+,+/+,NaN,9176,T,C,m.9176T>C
4,m.9176T>G,MT-ATP6,+/+,+/+,NaN,9176,T,G,m.9176T>G


shape: (88, 9)
missing variant_id: 0
unique variant_id: 88


# gnomAD

In [39]:
GNOMAD_VCF = list(RAW_DIR.glob("*.vcf.bgz"))[0]


print(GNOMAD_VCF)

/Users/dmitriiiliushchenko/PhD/Projects/mtdna_constraint/data/raw/gnomad.genomes.v3.1.sites.chrM.vcf.bgz


### Read VCF and write to pd.DataFrame

In [40]:
records = [] # records list

with open_vcf(GNOMAD_VCF) as f:
    for line in tqdm(f, desc="Parsing gnomAD VCF"): # See the progress
        if line.startswith("#"):
            continue
        
        chrom, pos, variant_id_vcf, ref, alt_raw, qual, filt, info = line.rstrip("\n").split("\t")[:8]
        info_dict = parse_info(info)
        alts = alt_raw.split(",")
        
        for i, alt in enumerate(alts):
            
            def allele_value(key):
                value = info_dict.get(key)
                
                if value is None:
                    return None
                
                value = str(value)
                
                # VEP/CSQ annotations do not separate by comma
                if "|" in value:
                    return value
                
                parts = value.split(",")
                
                if len(parts) == len(alts):
                    return parts[i]
                
                return value
            
            records.append({
                "chrom": chrom,
                "position": int(pos),
                "reference": ref.upper(),
                "alternate": alt.upper(),
                "variant_id": f"m.{pos}{ref.upper()}>{alt.upper()}",
                
                "gnomad_filter": filt,
                "AN": info_dict.get("AN"),
                
                "gnomad_homoplasmic_ac": allele_value("AC_hom"),
                "gnomad_heteroplasmic_ac": allele_value("AC_het"),
                "gnomad_homoplasmic_af": allele_value("AF_hom"),
                "gnomad_heteroplasmic_af": allele_value("AF_het"),
                "gnomad_max_heteroplasmy": allele_value("max_hl"),
                
                "hap_defining_variant": info_dict.get("hap_defining_variant"),
                "vep": info_dict.get("vep") or info_dict.get("VEP") or info_dict.get("CSQ"),
            })

gnomad = pd.DataFrame(records) # make pd

Parsing gnomAD VCF: 18226it [00:00, 54006.95it/s]


In [41]:
gnomad.tail()

,chrom,position,reference,alternate,variant_id,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_max_heteroplasmy,hap_defining_variant,vep
18159,chrM,16548,C,A,m.16548C>A,npg,56433,0,0,0.00000,0.00000,0.00000,None,A|intergenic_variant|MODIFIER|||Intergenic||||...
18160,chrM,16558,G,A,m.16558G>A,PASS,56434,1,0,1.77198e-05,0.00000,0.994000,None,A|intergenic_variant|MODIFIER|||Intergenic||||...
18161,chrM,16559,A,G,m.16559A>G,PASS,56433,3,0,5.31604e-05,0.00000,0.997000,None,G|intergenic_variant|MODIFIER|||Intergenic||||...
18162,chrM,16560,C,T,m.16560C>T,PASS,56434,1,0,1.77198e-05,0.00000,0.976000,None,T|intergenic_variant|MODIFIER|||Intergenic||||...
18163,chrM,16566,G,A,m.16566G>A,PASS,56434,7,0,0.000124039,0.00000,0.990000,None,A|intergenic_variant|MODIFIER|||Intergenic||||...


In [42]:
numeric_cols = [
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_max_heteroplasmy",
]

for col in numeric_cols:
    gnomad[col] = pd.to_numeric(gnomad[col], errors="coerce")

gnomad["gnomad_combined_af_simple"] = (
    gnomad["gnomad_homoplasmic_af"].fillna(0)
    + gnomad["gnomad_heteroplasmic_af"].fillna(0)
)

gnomad["gnomad_observed"] = 1

display(gnomad.head())
print("shape:", gnomad.shape)
print("unique variant_id:", gnomad["variant_id"].nunique())
print("unique positions:", gnomad["position"].nunique())

,chrom,position,reference,alternate,variant_id,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_max_heteroplasmy,hap_defining_variant,vep,gnomad_combined_af_simple,gnomad_observed
0,chrM,3,T,C,m.3T>C,PASS,56434,19,1,0.000337,0.000018,0.997,None,C|intergenic_variant|MODIFIER|||Intergenic||||...,0.000354,1
1,chrM,6,C,CCTCAA,m.6C>CCTCAA,npg,56433,0,0,0.000000,0.000000,0.000,None,CTCAA|intergenic_variant|MODIFIER|||Intergenic...,0.000000,1
2,chrM,7,A,G,m.7A>G,npg,56433,0,0,0.000000,0.000000,0.000,None,G|intergenic_variant|MODIFIER|||Intergenic||||...,0.000000,1
3,chrM,8,G,T,m.8G>T,PASS,56434,5,0,0.000089,0.000000,0.999,None,T|intergenic_variant|MODIFIER|||Intergenic||||...,0.000089,1
4,chrM,9,G,A,m.9G>A,PASS,56434,15,0,0.000266,0.000000,0.993,None,A|intergenic_variant|MODIFIER|||Intergenic||||...,0.000266,1


shape: (18164, 16)
unique variant_id: 18164
unique positions: 12749


Clean gnomAD and merge with masterr table

In [44]:
gnomad.head()

,chrom,position,reference,alternate,variant_id,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_max_heteroplasmy,hap_defining_variant,vep,gnomad_combined_af_simple,gnomad_observed
0,chrM,3,T,C,m.3T>C,PASS,56434,19,1,0.000337,0.000018,0.997,None,C|intergenic_variant|MODIFIER|||Intergenic||||...,0.000354,1
1,chrM,6,C,CCTCAA,m.6C>CCTCAA,npg,56433,0,0,0.000000,0.000000,0.000,None,CTCAA|intergenic_variant|MODIFIER|||Intergenic...,0.000000,1
2,chrM,7,A,G,m.7A>G,npg,56433,0,0,0.000000,0.000000,0.000,None,G|intergenic_variant|MODIFIER|||Intergenic||||...,0.000000,1
3,chrM,8,G,T,m.8G>T,PASS,56434,5,0,0.000089,0.000000,0.999,None,T|intergenic_variant|MODIFIER|||Intergenic||||...,0.000089,1
4,chrM,9,G,A,m.9G>A,PASS,56434,15,0,0.000266,0.000000,0.993,None,A|intergenic_variant|MODIFIER|||Intergenic||||...,0.000266,1


In [45]:
gnomad_clean = gnomad.copy()

gnomad_keep_cols = [
    "variant_id",
    "gnomad_observed",
    "gnomad_filter",
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "gnomad_max_heteroplasmy",
    "hap_defining_variant",
    "vep",
]

gnomad_keep_cols = [c for c in gnomad_keep_cols if c in gnomad_clean.columns]

gnomad_clean = gnomad_clean[gnomad_keep_cols].drop_duplicates("variant_id")

master = master.merge(
    gnomad_clean,
    on="variant_id",
    how="left"
)

master["gnomad_observed"] = master["gnomad_observed"].fillna(0).astype(int)

print(master.shape)
display(master.head())

(49704, 18)


,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Check how much positioons from gnomAD we have

In [46]:
master[master.gnomad_observed == 1]

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep
6,3,T,C,intergenic_variant,0.63872,m.3T>C,0.63872,1,PASS,56434.0,19.0,1.0,0.000337,0.000018,0.000354,0.997,None,C|intergenic_variant|MODIFIER|||Intergenic||||...
20,7,A,G,intergenic_variant,0.59916,m.7A>G,0.59916,1,npg,56433.0,0.0,0.0,0.000000,0.000000,0.000000,0.000,None,G|intergenic_variant|MODIFIER|||Intergenic||||...
21,8,G,T,intergenic_variant,0.58365,m.8G>T,0.58365,1,PASS,56434.0,5.0,0.0,0.000089,0.000000,0.000089,0.999,None,T|intergenic_variant|MODIFIER|||Intergenic||||...
25,9,G,A,intergenic_variant,0.57282,m.9G>A,0.57282,1,PASS,56434.0,15.0,0.0,0.000266,0.000000,0.000266,0.993,None,A|intergenic_variant|MODIFIER|||Intergenic||||...
27,10,T,C,intergenic_variant,0.56920,m.10T>C,0.56920,1,PASS,56434.0,11.0,0.0,0.000195,0.000000,0.000195,0.993,True,C|intergenic_variant|MODIFIER|||Intergenic||||...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49639,16548,C,A,intergenic_variant,0.74594,m.16548C>A,0.74594,1,npg,56433.0,0.0,0.0,0.000000,0.000000,0.000000,0.000,None,A|intergenic_variant|MODIFIER|||Intergenic||||...
49669,16558,G,A,intergenic_variant,0.77633,m.16558G>A,0.77633,1,PASS,56434.0,1.0,0.0,0.000018,0.000000,0.000018,0.994,None,A|intergenic_variant|MODIFIER|||Intergenic||||...
49673,16559,A,G,intergenic_variant,0.76800,m.16559A>G,0.76800,1,PASS,56433.0,3.0,0.0,0.000053,0.000000,0.000053,0.997,None,G|intergenic_variant|MODIFIER|||Intergenic||||...
49674,16560,C,T,intergenic_variant,0.76055,m.16560C>T,0.76055,1,PASS,56434.0,1.0,0.0,0.000018,0.000000,0.000018,0.976,None,T|intergenic_variant|MODIFIER|||Intergenic||||...


# HelixMTdb

In [47]:
HELIX_FILE = list(RAW_DIR.glob("*.tsv"))[0]

print(HELIX_FILE)

/Users/dmitriiiliushchenko/PhD/Projects/mtdna_constraint/data/raw/HelixMTdb_20200327.tsv


In [48]:
helix = read_table_tsv(HELIX_FILE)

display(helix.head())
print("shape:", helix.shape)
print(helix.columns.tolist())

,locus,alleles,feature,gene,counts_hom,af_hom,counts_het,af_het,mean_arf,max_arf,haplogroups_for_homoplasmic_variants,haplogroups_for_heteroplasmic_variants
0,chrM:5,"[""A"",""C""]",non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""H"",1]]",[]
1,chrM:10,"[""T"",""C""]",non_coding,MT-CRb,7,0.000036,1,0.000005,0.91892,0.91892,"[[""H"",7]]","[[""H"",1]]"
2,chrM:11,"[""C"",""T""]",non_coding,MT-CRb,0,0.000000,1,0.000005,0.60714,0.60714,[],"[[""H"",1]]"
3,chrM:12,"[""T"",""C""]",non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""D"",1]]",[]
4,chrM:16,"[""A"",""T""]",non_coding,MT-CRb,273,0.001393,4,0.000020,0.68971,0.92188,"[[""K"",246],[""U"",12],[""H"",7],[""V"",2],[""B"",1],[""...","[[""K"",2],[""L3"",1],[""U"",1]]"


shape: (14324, 12)
['locus', 'alleles', 'feature', 'gene', 'counts_hom', 'af_hom', 'counts_het', 'af_het', 'mean_arf', 'max_arf', 'haplogroups_for_homoplasmic_variants', 'haplogroups_for_heteroplasmic_variants']


### Get variantID from HeliMTxdb

In [49]:
import ast

helix_clean = helix.copy()

helix_clean["position"] = (
    helix_clean["locus"]
    .astype(str)
    .str.extract(r"chrM:(\d+)", expand=False)
)

helix_clean["position"] = pd.to_numeric(helix_clean["position"], errors="coerce").astype("Int64")


def parse_helix_alleles(x):
    if pd.isna(x):
        return pd.Series({"reference": pd.NA, "alternate": pd.NA})
    
    try:
        alleles = ast.literal_eval(str(x))
        return pd.Series({
            "reference": str(alleles[0]).upper(),
            "alternate": str(alleles[1]).upper(),
        })
    except Exception:
        found = re.findall(r"[ACGT]", str(x).upper())
        
        if len(found) >= 2:
            return pd.Series({
                "reference": found[0],
                "alternate": found[1],
            })
        
        return pd.Series({"reference": pd.NA, "alternate": pd.NA})


helix_clean[["reference", "alternate"]] = helix_clean["alleles"].apply(parse_helix_alleles)

helix_clean["variant_id"] = (
    "m."
    + helix_clean["position"].astype(str)
    + helix_clean["reference"]
    + ">"
    + helix_clean["alternate"]
)

display(helix_clean.head())

,locus,alleles,feature,gene,counts_hom,af_hom,counts_het,af_het,mean_arf,max_arf,haplogroups_for_homoplasmic_variants,haplogroups_for_heteroplasmic_variants,position,reference,alternate,variant_id
0,chrM:5,"[""A"",""C""]",non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""H"",1]]",[],5,A,C,m.5A>C
1,chrM:10,"[""T"",""C""]",non_coding,MT-CRb,7,0.000036,1,0.000005,0.91892,0.91892,"[[""H"",7]]","[[""H"",1]]",10,T,C,m.10T>C
2,chrM:11,"[""C"",""T""]",non_coding,MT-CRb,0,0.000000,1,0.000005,0.60714,0.60714,[],"[[""H"",1]]",11,C,T,m.11C>T
3,chrM:12,"[""T"",""C""]",non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""D"",1]]",[],12,T,C,m.12T>C
4,chrM:16,"[""A"",""T""]",non_coding,MT-CRb,273,0.001393,4,0.000020,0.68971,0.92188,"[[""K"",246],[""U"",12],[""H"",7],[""V"",2],[""B"",1],[""...","[[""K"",2],[""L3"",1],[""U"",1]]",16,A,T,m.16A>T


In [50]:
helix_clean = helix_clean.rename(columns={
    "counts_hom": "helix_counts_hom",
    "af_hom": "helix_af_hom",
    "counts_het": "helix_counts_het",
    "af_het": "helix_af_het",
    "mean_arf": "helix_mean_arf",
    "max_arf": "helix_max_arf",
    "haplogroups_for_homoplasmic_variants": "helix_haplogroups_hom",
    "haplogroups_for_heteroplasmic_variants": "helix_haplogroups_het",
})

helix_keep_cols = [
    "variant_id",
    "feature",
    "gene",
    "helix_counts_hom",
    "helix_af_hom",
    "helix_counts_het",
    "helix_af_het",
    "helix_mean_arf",
    "helix_max_arf",
    "helix_haplogroups_hom",
    "helix_haplogroups_het",
]

helix_keep_cols = [c for c in helix_keep_cols if c in helix_clean.columns]

helix_clean = helix_clean[helix_keep_cols].drop_duplicates("variant_id")
display(helix_clean.head())

,variant_id,feature,gene,helix_counts_hom,helix_af_hom,helix_counts_het,helix_af_het,helix_mean_arf,helix_max_arf,helix_haplogroups_hom,helix_haplogroups_het
0,m.5A>C,non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""H"",1]]",[]
1,m.10T>C,non_coding,MT-CRb,7,0.000036,1,0.000005,0.91892,0.91892,"[[""H"",7]]","[[""H"",1]]"
2,m.11C>T,non_coding,MT-CRb,0,0.000000,1,0.000005,0.60714,0.60714,[],"[[""H"",1]]"
3,m.12T>C,non_coding,MT-CRb,1,0.000005,0,0.000000,NaN,NaN,"[[""D"",1]]",[]
4,m.16A>T,non_coding,MT-CRb,273,0.001393,4,0.000020,0.68971,0.92188,"[[""K"",246],[""U"",12],[""H"",7],[""V"",2],[""B"",1],[""...","[[""K"",2],[""L3"",1],[""U"",1]]"


In [51]:
ds3 = ds3.copy()

ds3["variant_id"] = (
    "m."
    + ds3["position"].astype(str)
    + ds3["reference"]
    + ">"
    + ds3["alternate"]
)


print(ds3.shape)
display(ds3.head())

(171, 5)


,position,reference,alternate,classification_group,variant_id
0,3385,A,T,VUS of low clinical significance,m.3385A>T
1,3391,G,A,VUS of low clinical significance,m.3391G>A
2,3398,T,C,Benign & Likely Benign,m.3398T>C
3,3460,G,A,Pathogenic & Likely pathogenic,m.3460G>A
4,3497,C,T,Benign & Likely Benign,m.3497C>T


In [52]:
ds3_label = ds3[["variant_id"]].drop_duplicates().copy()
ds3_label["is_disease_suspected_dataset3"] = 1

master = master.merge(
    ds3_label,
    on="variant_id",
    how="left"
)

master["is_disease_suspected_dataset3"] = (
    master["is_disease_suspected_dataset3"]
    .fillna(0)
    .astype(int)
)
display(master)

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep,is_disease_suspected_dataset3
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49699,16568,T,A,intergenic_variant,0.67690,m.16568T>A,0.67690,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
49700,16568,T,G,intergenic_variant,0.67690,m.16568T>G,0.67690,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
49701,16569,G,T,intergenic_variant,0.66691,m.16569G>T,0.66691,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
49702,16569,G,A,intergenic_variant,0.66691,m.16569G>A,0.66691,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [53]:
ds8_label = ds8[["variant_id"]].drop_duplicates().copy()
ds8_label["is_neutral_dataset8"] = 1

master = master.merge(
    ds8_label,
    on="variant_id",
    how="left"
)

master["is_neutral_dataset8"] = (
    master["is_neutral_dataset8"]
    .fillna(0)
    .astype(int)
)

In [54]:
ds9_label = ds9[["variant_id"]].drop_duplicates().copy()
ds9_label["is_pathogenic_dataset9"] = 1

master = master.merge(
    ds9_label,
    on="variant_id",
    how="left"
)

master["is_pathogenic_dataset9"] = (
    master["is_pathogenic_dataset9"]
    .fillna(0)
    .astype(int)
)

In [55]:
master["validation_label"] = "unlabeled"

master.loc[
    master["is_neutral_dataset8"].eq(1),
    "validation_label"
] = "neutral"

master.loc[
    master["is_disease_suspected_dataset3"].eq(1),
    "validation_label"
] = "disease_suspected"

master.loc[
    master["is_pathogenic_dataset9"].eq(1),
    "validation_label"
] = "pathogenic_confirmed"

master["validation_label"].value_counts(dropna=False)

validation_label
unlabeled               41280
neutral                  8183
disease_suspected         153
pathogenic_confirmed       88
Name: count, dtype: int64

In [56]:
def annotate_intervals_by_position(master, intervals, prefix):
    intervals = intervals.copy()
    
    intervals["start_position"] = pd.to_numeric(intervals["start_position"], errors="coerce")
    
    intervals["end_position"] = pd.to_numeric(intervals["end_position"], errors="coerce")
    
    interval_cols = intervals.columns.tolist()
    
    rows = []
    
    for _, row in intervals.iterrows():
        mask = (master["position"].ge(row["start_position"]) & master["position"].le(row["end_position"]))
        
        tmp = master.loc[mask, ["variant_id"]].copy()
        
        for col in interval_cols:
            tmp[f"{prefix}_{col}"] = row[col]
        
        rows.append(tmp)
    
    if len(rows) == 0:
        return master
    
    annotation = pd.concat(rows, ignore_index=True)
    annotation = annotation.drop_duplicates("variant_id")
    
    out = master.merge(annotation, on="variant_id", how="left")
    
    return out

In [57]:
master = annotate_intervals_by_position(master=master, intervals=ds1, prefix="gene_constraint")

print(master.shape)

(49704, 31)


In [58]:
master = annotate_intervals_by_position(master=master, intervals=ds2, prefix="regional_constraint")

master["in_regional_constraint"] = master["regional_constraint_start_position"].notna().astype(int)

print(master.shape)

(49704, 42)


In [59]:
master = annotate_intervals_by_position(master=master, intervals=ds5, prefix="noncoding_constraint")

master["in_noncoding_constraint"] = master["noncoding_constraint_start_position"].notna().astype(int)

print(master.shape)

(49704, 52)


In [61]:
artifact_sites = [301, 302, 310, 316, 3107, 16182]

master["is_artifact_prone_site"] = master["position"].isin(artifact_sites).astype(int)

In [62]:
master.head()

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,...,noncoding_constraint_description,noncoding_constraint_start_position,noncoding_constraint_end_position,noncoding_constraint_observed,noncoding_constraint_expected,noncoding_constraint_obs_exp,noncoding_constraint_lower_ci,noncoding_constraint_upper_ci,in_noncoding_constraint,is_artifact_prone_site
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
